In [7]:
import pandas as pd
import re
import unicodedata

In [22]:
df_authors = pd.read_csv("data/entities/autores_ecuador_enriquecido.csv")
print("Forma de df_authors:", df_authors.shape)
df_authors.head()

Forma de df_authors: (65547, 9)


,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']",20.0,"['Computer Science Applications', 'Astronomy a..."
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598'],2.0,"['Computational Theory and Mathematics', 'Comp..."
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322...",11.0,"['Ecology', 'Business, Management and Accounti..."
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9.0,"['Immunology', 'Medicine (all)', 'Earth and Pl..."
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063'],2.0,['Plant Science']


In [23]:
df_auths_labeled = pd.read_csv("data/labels/authors_with_gender_improved.csv")
print("Forma de df_auths_labeled:", df_auths_labeled.shape)
df_auths_labeled.head()

Forma de df_auths_labeled: (39225, 6)


,identifier,first_name,last_name,auth_name,initials,gender
0,57219054382,Jorge Edwin,Ormaza Andrade,Ormaza Andrade J.E.,J.E.,male
1,57192930404,Mario,Hurtado,Hurtado M.,M.,male
2,57192803433,Ruth Elizabeth,Minga-Vallejo,Minga-Vallejo R.E.,R.E.,unknown
3,57220465983,Carlos,Tapia,Tapia C.,C.,male
4,57215549671,Pelayo,Salinas-DeLeón,Salinas-DeLeón P.,P.,male


In [24]:
# IDs que estan en ambos df
ids_coincidentes = set(df_authors["authid"]).intersection(
    set(df_auths_labeled["identifier"])
)

print(f"Total autores en df_authors: {len(df_authors)}")
print(f"Total autores etiquetados disponibles: {len(df_auths_labeled)}")
print(f"Autores coincidentes encontrados: {len(ids_coincidentes)}")

Total autores en df_authors: 65547
Total autores etiquetados disponibles: 39225
Autores coincidentes encontrados: 35857


In [25]:
ids_authors = set(df_authors["authid"])  # IDs sin genero
ids_labeled = set(df_auths_labeled["identifier"])  # IDs con genero

# IDs que estan en ambos df
ids_coincidentes = ids_authors.intersection(ids_labeled)
# Diferencia
ids_inconsistentes = ids_labeled - ids_authors

print(f"Total autores en df_authors: {len(df_authors)}")
print(f"Total autores etiquetados disponibles: {len(df_auths_labeled)}")
print(f"Total autores coincidentes: {len(ids_coincidentes)}")
print(f"Total autores con IDs inconsistentes: {len(ids_inconsistentes)}")

Total autores en df_authors: 65547
Total autores etiquetados disponibles: 39225
Total autores coincidentes: 35857
Total autores con IDs inconsistentes: 3368


In [26]:
df_autores_inconsistentes = df_auths_labeled[
    df_auths_labeled["identifier"].isin(ids_inconsistentes)
].copy()

df_autores_inconsistentes.head()

,identifier,first_name,last_name,auth_name,initials,gender
10,57204354637,Pablo,Garcés,Garcés P.,P.,male
14,57195804959,L.,Carrión,Carrión L.,L.,unknown
34,57363577200,Lenin Carlos,Gabriel Flores,Gabriel Flores L.C.,L.C.,male
53,57686848600,Beatriz,Loor-Avila,Loor-Avila B.,B.,female
69,57659590000,Roberto Andrés García,Viteri,Viteri R.A.G.,R.A.G.,male


In [27]:
df_tmp = df_authors[df_authors["authname"].isin(df_autores_inconsistentes["auth_name"].tolist())]
df_tmp

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas
26,11339147200,Mosquera J.,Mosquera,J.,J.,NaN,"['105333303', '60072038']",2.0,"['Veterinary (all)', 'Pediatrics, Perinatology..."
28,11340096600,Córdova M.,Córdova,Maria E.,M.E.,NaN,['105333303'],1.0,"['Immunology', 'Microbiology', 'Virology', 'In..."
61,12446067900,Játiva-Mariño E.,Játiva-Mariño,Edgar,E.,0000-0002-6658-0779,"['100464803', '60072038']",6.0,"['Medicine (miscellaneous)', 'Medicine (all)',..."
75,12770332900,Vásconez J.,Vásconez,Juan,J.,NaN,['102008337'],1.0,['Animal Science and Zoology']
78,12772151000,Reyes M.,Reyes,Monica,M.,NaN,['60072059'],2.0,"['Food Science', 'Medicine (miscellaneous)', '..."
...,...,...,...,...,...,...,...,...,...
65462,8720668000,Sánchez D.,Sánchez,Dora,D.,NaN,"['100877019', '106173420', '60072048']",8.0,"['Anatomy', 'Biochemistry, Genetics and Molecu..."
65503,8980750500,Baca M.,Baca,Martin,M.,NaN,['60072059'],27.0,"['Theoretical Computer Science', 'Electrical a..."
65512,9243365600,Torres R.,Torres,Rene,R.,NaN,['60072059'],4.0,"['Animal Science and Zoology', 'Aquatic Scienc..."
65521,9338367100,Vera L.,Vera,Leonardo,L.,NaN,['60116971'],6.0,['Political Science and International Relation...


In [12]:
df_authors[df_authors["authname"].isin(["Garcés P.", "Carrión L.", "Gabriel Flores L.C.", "Loor-Avila B.", "Viteri R.A.G."])]

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas
6244,57188836684,Carrión L.,Carrión,Luis,L.,0000-0002-6069-7888,['60104598'],6.0,"['Physics and Astronomy (all)', 'Electrical an..."
19913,57212142255,Loor-Avila B.,Loor-Avila,Beatriz,B.,NaN,"['60072050', '60104602', '60110590']",1.0,"['Management of Technology and Innovation', 'E..."
23986,57218549251,Viteri R.A.G.,Viteri,Roberto Andrés García,R.A.G.,0000-0002-6096-9628,"['60072042', '60110590']",1.0,"['Computer Science (all)', 'Business, Manageme..."


In [28]:
subset_labeled = df_auths_labeled[["identifier", "gender"]]

# Realizamos el merge
df_final = pd.merge(
    df_authors,
    subset_labeled,
    left_on="authid",  # Clave en la tabla izquierda
    right_on="identifier",  # Clave en la tabla derecha
    how="left",  # Mantiene estructura de la izquierda
)

# Rellenar los valores nulos con "unknown"
# df_final["gender"] = df_final["gender"].fillna("unknown")

# Eliminar la columna "identifier"
df_final.drop(columns=["identifier"], inplace=True)

# Verificamos el resultado
print("Forma del nuevo dataframe:", df_final.shape)
df_final.head()

Forma del nuevo dataframe: (65547, 10)


,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']",20.0,"['Computer Science Applications', 'Astronomy a...",unknown
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598'],2.0,"['Computational Theory and Mathematics', 'Comp...",male
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322...",11.0,"['Ecology', 'Business, Management and Accounti...",unknown
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9.0,"['Immunology', 'Medicine (all)', 'Earth and Pl...",female
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063'],2.0,['Plant Science'],male


In [ ]:
# Contar cuántos nulos quedaron en la columna gender
nulos = df_final["gender"].isnull().sum()
completos = df_final["gender"].notnull().sum()

print(f"Autores con género asignado: {completos}")
print(f"Autores sin información de género (NaN): {nulos}")
print(f"Porcentaje completado: {completos / len(df_final) * 100:.2f}%")

Autores con género asignado: 35857
Autores sin información de género (NaN): 29690
Porcentaje completado: 54.70%


In [30]:
df_autores_sin_genero = df_final[
    ~df_final["authid"].isin(ids_labeled)
].copy()

df_autores_sin_genero

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender
7,10639679200,Roy S.,Roy,Sandip,S.,NaN,['60072054'],27.0,"['Biochemistry, Genetics and Molecular Biology...",NaN
12,11238949500,Alvarez H.H.,Alvarez,Hilda Hernández,H.H.,0000-0002-5596-7644,['60072042'],10.0,"['Plant Science', 'Microbiology (medical)', 'A...",NaN
53,12243905200,Yadama V.,Yadama,Vikram,V.,NaN,['60072035'],17.0,"['Organic Chemistry', 'Energy (all)', 'Environ...",NaN
70,12769225100,Hansoti B.,Hansoti,Bhakti,B.,NaN,['131486340'],22.0,"['Health (social science)', 'Oncology', 'Compu...",NaN
96,12807476200,Zurita Altamirano J.,Zurita Altamirano,Julio,J.,0000-0003-0591-7371,['60072063'],0.0,"['Business, Management and Accounting (all)', ...",NaN
...,...,...,...,...,...,...,...,...,...,...
65505,8987177900,Garrido M.S.,Garrido,María Sol,M.S.,NaN,['102050610'],0.0,['Medicine (all)'],NaN
65507,8988890100,Svensson J.T.,Svensson,Jan T.,J.T.,NaN,['60072061'],26.0,"['Biotechnology', 'Genetics (clinical)', 'Food...",NaN
65510,9044317400,Zuniga M.,Zuniga,Marco,M.,NaN,['60105693'],23.0,"['Hardware and Architecture', 'Computer Vision...",NaN
65512,9243365600,Torres R.,Torres,Rene,R.,NaN,['60072059'],4.0,"['Animal Science and Zoology', 'Aquatic Scienc...",NaN


In [31]:
df_autores_con_genero = df_final[
    df_final["authid"].isin(ids_labeled)
].copy()

df_autores_con_genero

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']",20.0,"['Computer Science Applications', 'Astronomy a...",unknown
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598'],2.0,"['Computational Theory and Mathematics', 'Comp...",male
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322...",11.0,"['Ecology', 'Business, Management and Accounti...",unknown
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9.0,"['Immunology', 'Medicine (all)', 'Earth and Pl...",female
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063'],2.0,['Plant Science'],male
...,...,...,...,...,...,...,...,...,...,...
65542,9841468300,Guijarro M.J.M.,Guijarro,María José Muñoz,M.J.M.,NaN,['60072063'],1.0,"['Social Sciences (all)', 'Genetics (clinical)...",female
65543,9941450100,Mestanza-Peralta M.A.,Mestanza-Peralta,M. A.,M.A.,NaN,['107662971'],5.0,"['Rheumatology', 'Orthopedics and Sports Medic...",unknown
65544,9942096300,Niebieskikwiat D.,Niebieskikwiat,Darío,D.,0000-0001-7591-9319,['60072059'],18.0,"['Organic Chemistry', 'Surfaces, Coatings and ...",male
65545,9942296700,Sánchez-Urdaneta A.B.,Sánchez-Urdaneta,Adriana Beatriz,A.B.,0000-0003-3108-0296,['60108912'],10.0,"['Nutrition and Dietetics', 'Ecology', 'Earth ...",female


In [32]:
df_autores_sin_genero.to_csv("data/labels/autores_sin_genero.csv", index=False)

In [33]:
df_autores_con_genero.to_csv("data/labels/autores_con_genero.csv", index=False)

In [34]:
ids_labeled = set(df_auths_labeled["identifier"]) # IDs con genero
ids_authors = set(df_authors["authid"])           # IDs sin genero

# Calcular la diferencia
ids_no_presentes = ids_labeled - ids_authors

print(f"Total IDs en labeled que sobran (no están en authors): {len(ids_no_presentes)}")
print("Muestra de 5 IDs faltantes:", list(ids_no_presentes)[:5])

Total IDs en labeled que sobran (no están en authors): 3368
Muestra de 5 IDs faltantes: [57212010496, 57213411330, 57718587400, 57200246793, 57216409609]


In [9]:
df_auths_labeled[df_auths_labeled["identifier"] == 57212010496]

,identifier,first_name,last_name,auth_name,initials,gender
16769,57212010496,Johanna,Castillo-Cabrera,Castillo-Cabrera J.,J.,female


# Expand genders

In [35]:
df_auths_with_genders = pd.read_csv("data/labels/autores_con_genero.csv")
df_auths_with_genders.head(10)

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']",20.0,"['Computer Science Applications', 'Astronomy a...",unknown
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598'],2.0,"['Computational Theory and Mathematics', 'Comp...",male
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322...",11.0,"['Ecology', 'Business, Management and Accounti...",unknown
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9.0,"['Immunology', 'Medicine (all)', 'Earth and Pl...",female
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063'],2.0,['Plant Science'],male
5,10539604000,Pagán-Jiménez J.R.,Pagán-Jiménez,Jaime R.,J.R.,0000-0001-9613-588X,['115015256'],13.0,"['Global and Planetary Change', 'Earth-Surface...",unknown
6,10639110600,Rodriguez R.O.,Rodriguez,R. O.,R.O.,NaN,['60108603'],5.0,"['Nuclear and High Energy Physics', 'Physics a...",unknown
7,10838880900,Morocho N.,Morocho,Nancy,N.,NaN,"['100556385', '100774304', '60072050']",7.0,"['Endocrinology', 'Obstetrics and Gynecology',...",unknown
8,10840143000,Bayas A.,Bayas,Antonio,A.,NaN,['60072054'],1.0,"['Electrical and Electronic Engineering', 'Com...",male
9,11141724800,Smalligan R.,Smalligan,Roger,R.,NaN,['60072047'],13.0,"['Leadership and Management', 'Epidemiology', ...",male


In [38]:
df_auths_with_genders["gender"].value_counts()

gender
male       17584
female      9785
unknown     8488
Name: count, dtype: int64

In [39]:
df_auths_with_genders[df_auths_with_genders["gender"].isin(["female"])]

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9.0,"['Immunology', 'Medicine (all)', 'Earth and Pl...",female
13,11239212800,Guzmán A.,Guzmán,Anamaría,A.,NaN,['60072034'],0.0,"['Neuroscience (all)', 'Neurology (clinical)']",female
18,11239593000,Fernando Estévez A.,Fernando Estévez,A.,A.,NaN,['100520461'],0.0,"['Neurology (clinical)', 'Neurology', 'Neurosc...",female
21,11240223600,Herrera D.,Herrera,Dora,D.,NaN,['122559898'],16.0,"['Anthropology', 'Developmental Neuroscience',...",female
26,11340096600,Córdova M.,Córdova,Maria E.,M.E.,NaN,['105333303'],1.0,"['Immunology', 'Microbiology', 'Virology', 'In...",female
...,...,...,...,...,...,...,...,...,...,...
35837,9633702000,Chang C.,Chang,Carolina,C.,NaN,['114459557'],6.0,"['Control and Systems Engineering', 'Computer ...",female
35838,9638630100,Martínez P.C.,Martínez,Priscilla C.,P.C.,NaN,"['106440381', '60072046']",3.0,"['Oceanography', 'Ecology', 'Nature and Landsc...",female
35851,9839762300,Celi A.P.,Celi,Ana P.,A.P.,NaN,['60104598'],1.0,"['Genetics (clinical)', 'Social Sciences (all)...",female
35852,9841468300,Guijarro M.J.M.,Guijarro,María José Muñoz,M.J.M.,NaN,['60072063'],1.0,"['Social Sciences (all)', 'Genetics (clinical)...",female


In [40]:
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

In [41]:
_initial_like = re.compile(r"^([A-Za-z]\.)+$|^[A-Za-z]$|^[A-Za-z]{1,2}$")

In [42]:
def clean_given_name(given):
    if pd.isna(given):
        return ""
    
    s = str(given).strip()
    if not s:
        return ""

    s = s.lower()

    s = s.replace("-", " ")
    # elimina caracteres raros excepto letras, espacios, puntos y acentos
    s = re.sub(r"[^\w\s\.\u00C0-\u017F]", " ", s)
    s = re.sub(r"\s+", " ", s)
    tokens = []
    for t in s.split():
        t = t.strip()
        if not t:
            continue

        # descarta tokens tipo inicial
        if _initial_like.match(strip_accents(t)):
            continue

        # descarta tokens de longitud 1
        if len(strip_accents(t)) <= 1:
            continue

        # descarta "jr/sr" si aparece
        if strip_accents(t) in {"jr", "sr"}:
            continue

        tokens.append(t)

    return " ".join(tokens).strip()

In [43]:
def normalize_key(s):
    s = strip_accents(s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [44]:
df_auths_without_gender = pd.read_csv("data/labels/autores_sin_genero.csv")
df_auths_without_gender.head()

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender
0,10639679200,Roy S.,Roy,Sandip,S.,NaN,['60072054'],27.0,"['Biochemistry, Genetics and Molecular Biology...",NaN
1,11238949500,Alvarez H.H.,Alvarez,Hilda Hernández,H.H.,0000-0002-5596-7644,['60072042'],10.0,"['Plant Science', 'Microbiology (medical)', 'A...",NaN
2,12243905200,Yadama V.,Yadama,Vikram,V.,NaN,['60072035'],17.0,"['Organic Chemistry', 'Energy (all)', 'Environ...",NaN
3,12769225100,Hansoti B.,Hansoti,Bhakti,B.,NaN,['131486340'],22.0,"['Health (social science)', 'Oncology', 'Compu...",NaN
4,12807476200,Zurita Altamirano J.,Zurita Altamirano,Julio,J.,0000-0003-0591-7371,['60072063'],0.0,"['Business, Management and Accounting (all)', ...",NaN


In [45]:
df_auths_with_genders["given_clean"] = df_auths_with_genders["given-name"].apply(clean_given_name)
df_auths_with_genders["given_key"] = df_auths_with_genders["given_clean"].apply(normalize_key)
df_auths_with_genders.head(20)

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender,given_clean,given_key
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']",20.0,"['Computer Science Applications', 'Astronomy a...",unknown,,
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598'],2.0,"['Computational Theory and Mathematics', 'Comp...",male,carlos,carlos
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322...",11.0,"['Ecology', 'Business, Management and Accounti...",unknown,víctor,victor
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9.0,"['Immunology', 'Medicine (all)', 'Earth and Pl...",female,risa,risa
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063'],2.0,['Plant Science'],male,santiago,santiago
5,10539604000,Pagán-Jiménez J.R.,Pagán-Jiménez,Jaime R.,J.R.,0000-0001-9613-588X,['115015256'],13.0,"['Global and Planetary Change', 'Earth-Surface...",unknown,jaime,jaime
6,10639110600,Rodriguez R.O.,Rodriguez,R. O.,R.O.,NaN,['60108603'],5.0,"['Nuclear and High Energy Physics', 'Physics a...",unknown,,
7,10838880900,Morocho N.,Morocho,Nancy,N.,NaN,"['100556385', '100774304', '60072050']",7.0,"['Endocrinology', 'Obstetrics and Gynecology',...",unknown,nancy,nancy
8,10840143000,Bayas A.,Bayas,Antonio,A.,NaN,['60072054'],1.0,"['Electrical and Electronic Engineering', 'Com...",male,antonio,antonio
9,11141724800,Smalligan R.,Smalligan,Roger,R.,NaN,['60072047'],13.0,"['Leadership and Management', 'Epidemiology', ...",male,roger,roger


In [46]:
labeled = df_auths_with_genders[df_auths_with_genders["gender"].isin(["male", "female"]) & df_auths_with_genders["given_key"].ne("")].copy()

In [ ]:
name_sets = labeled.groupby("given_key")["gender"].agg(lambda x: set(x))
ok = name_sets[name_sets.apply(len) == 1]
ambiguous = set(name_sets[name_sets.apply(len) > 1].index) # mismo given_key con ambos generos
print(f"Nombres únicos con género consistente: {len(ok)}")
print(f"Nombres ambiguos (género inconsistente): {len(ambiguous)}")

Nombres únicos con género consistente: 10236
Nombres ambiguos (género inconsistente): 53


In [49]:
print(ambiguous)

{'fernanda', 'gladys', 'bernarda', 'hernandez', 'garcia', 'jorge', 'diego', 'francisco', 'lozada', 'rosario', 'ortiz', 'lourdes', 'belen', 'carolina', 'susana', 'estefania', 'sofia', 'leonardo', 'marco antonio', 'christian', 'darwin', 'yadira', 'edison', 'karina', 'maria del rosario', 'danilo', 'robin', 'mejia', 'ximena', 'ronald', 'martha elizabeth', 'eduardo', 'silvia', 'glenda', 'bravo', 'mercedes', 'vivian', 'paul', 'roberto', 'gustavo', 'ahmed', 'cynthia', 'david', 'efren', 'toapanta', 'flavio', 'gonzalo', 'moya', 'daniela', 'viviana', 'paola', 'jose david', 'rolando'}


In [57]:
df_auths_with_genders[df_auths_with_genders["given-name"].str.contains("marco antonio", case=False, na=False)]

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender,given_clean,given_key
3437,55974296000,López-Águila M.A.,López-Águila,Marco Antonio,M.A.,NaN,['60108603'],1.0,"['Neuroscience (all)', 'Neuropsychology and Ph...",male,marco antonio,marco antonio
14321,57204711547,Villalva M.A.,Villalva,Marco Antonio,M.A.,NaN,['60072042'],0.0,"['Computer Networks and Communications', 'Arti...",male,marco antonio,marco antonio
15950,57208186302,Chauvin M.A.A.,Chauvin,Marco Antonio Ayala,M.A.A.,0000-0002-0084-6773,['60072064'],1.0,"['Health Policy', 'Education', 'Public Health,...",male,marco antonio ayala,marco antonio ayala
16658,57209505549,Arauz M.A.H.,Arauz,Marco Antonio Hernández,M.A.H.,0000-0002-8476-0896,"['60104598', '60110666']",1.0,"['Mathematics (all)', 'Multidisciplinary', 'Me...",male,marco antonio hernández,marco antonio hernandez
18808,57211910148,Veloz Jaramillo M.A.,Veloz Jaramillo,Marco Antonio,M.A.,0000-0002-3178-7278,"['60104598', '60109986']",1.0,"['Numerical Analysis', 'Applied Mathematics', ...",female,marco antonio,marco antonio
21806,57216897315,Criollo M.A.,Criollo,Marco Antonio,M.A.,NaN,['124448458'],1.0,"['Decision Sciences (all)', 'Computer Science ...",male,marco antonio,marco antonio
22410,57217987165,Salas Subía M.A.,Salas Subía,Marco Antonio,M.A.,0000-0003-2411-2325,['124759059'],1.0,['Social Sciences (all)'],male,marco antonio,marco antonio
24739,57220804101,Checa Cabrera M.A.,Checa Cabrera,Marco Antonio,M.A.,0000-0002-4169-581X,"['123346252', '60113393']",3.0,"['Education', 'Applied Mathematics', 'Social S...",male,marco antonio,marco antonio
25645,57221983571,Calle Gómez M.A.,Calle Gómez,Marco Antonio,M.A.,0000-0002-2706-1554,"['60072042', '60072050']",1.0,"['History', 'Business, Management and Accounti...",male,marco antonio,marco antonio
25792,57221999673,Calle Prado M.A.,Calle Prado,Marco Antonio,M.A.,0000-0002-1133-8336,['60072042'],1.0,"['Education', 'Social Sciences (all)', 'Econom...",male,marco antonio,marco antonio


In [58]:
name_to_gender = ok.apply(lambda s: next(iter(s))).to_dict()

In [60]:
df_auths_without_gender["given_clean"] = df_auths_without_gender["given-name"].apply(clean_given_name)
df_auths_without_gender["given_key"] = df_auths_without_gender["given_clean"].apply(normalize_key)

In [61]:
mask = (
    df_auths_without_gender["given_key"].ne("")
    & ~df_auths_without_gender["given_key"].isin(ambiguous)
    & df_auths_without_gender["given_key"].isin(name_to_gender)
)

df_auths_without_gender.loc[mask, "gender"] = df_auths_without_gender.loc[mask, "given_key"].map(name_to_gender)

C:\Users\ADCJ\AppData\Local\Temp\ipykernel_14056\1696483731.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['male' 'female' 'male' ... 'female' 'male' 'male']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_auths_without_gender.loc[mask, "gender"] = df_auths_without_gender.loc[mask, "given_key"].map(name_to_gender)


In [62]:
print("Imputados en target:", mask.sum())
print(df_auths_without_gender["gender"].value_counts(dropna=False))

Imputados en target: 12395
gender
NaN       17295
male       7881
female     4514
Name: count, dtype: int64


In [64]:
df_auths_without_gender[df_auths_without_gender["gender"].notna()].head(20)

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender,given_clean,given_key
4,12807476200,Zurita Altamirano J.,Zurita Altamirano,Julio,J.,0000-0003-0591-7371,['60072063'],0.0,"['Business, Management and Accounting (all)', ...",male,julio,julio
5,13407130900,Quintero B.,Quintero,Beatriz,B.,0000-0002-1084-9185,['60072064'],7.0,"['Microbiology (medical)', 'Food Science', 'In...",female,beatriz,beatriz
7,14039344200,Saladié Ò.,Saladié,Òscar,Ò.,NaN,['60072061'],14.0,"['Global and Planetary Change', 'Waste Managem...",male,òscar,oscar
11,14825183400,Mideros D.,Mideros,Daniel,D.,NaN,['60108339'],5.0,"['Electronic, Optical and Magnetic Materials',...",male,daniel,daniel
12,15021299100,Stewart J.R.,Stewart,Jill R.,J.R.,NaN,['60072059'],33.0,"['Organic Chemistry', 'Pediatrics, Perinatolog...",male,jill,jill
15,15049769800,Suárez J.C.,Suárez,Juan Carlos,J.C.,NaN,['60072061'],12.0,"['Mathematics (all)', 'Oceanography', 'Surface...",male,juan carlos,juan carlos
17,15126496200,García-Navarro J.,García-Navarro,Justo,J.,NaN,['60072035'],14.0,"['Engineering (miscellaneous)', 'Environmental...",male,justo,justo
18,15520456600,Caicedo B.,Caicedo,Bernardo,B.,NaN,['60072059'],24.0,"['Mechanical Engineering', 'Geochemistry and P...",male,bernardo,bernardo
20,16067849900,Barra W.,Barra,Walter,W.,NaN,['60072061'],4.0,"['Computer Science (all)', 'Materials Science ...",male,walter,walter
21,16229879700,Hernández-Hernández R.,Hernández-Hernández,Rafael,R.,NaN,['60108339'],28.0,"['Community and Home Care', 'Pharmacology', 'P...",male,rafael,rafael


In [78]:
tmp_cols = ["given_clean", "given_key"]
df_auths_with_genders = df_auths_with_genders.drop(columns=tmp_cols, errors="ignore")
df_auths_without_gender = df_auths_without_gender.drop(columns=tmp_cols, errors="ignore")

df_imputed = df_auths_without_gender[df_auths_without_gender["gender"].notna()].copy()
df_genders_incomputed = df_auths_without_gender[df_auths_without_gender["gender"].isna()].copy()

df_final = pd.concat([df_auths_with_genders, df_imputed], ignore_index=True)
print("Forma del dataframe final:", df_final.shape)
df_final.head(15)

Forma del dataframe final: (48252, 10)


,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas,gender
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']",20.0,"['Computer Science Applications', 'Astronomy a...",unknown
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598'],2.0,"['Computational Theory and Mathematics', 'Comp...",male
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322...",11.0,"['Ecology', 'Business, Management and Accounti...",unknown
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9.0,"['Immunology', 'Medicine (all)', 'Earth and Pl...",female
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063'],2.0,['Plant Science'],male
5,10539604000,Pagán-Jiménez J.R.,Pagán-Jiménez,Jaime R.,J.R.,0000-0001-9613-588X,['115015256'],13.0,"['Global and Planetary Change', 'Earth-Surface...",unknown
6,10639110600,Rodriguez R.O.,Rodriguez,R. O.,R.O.,NaN,['60108603'],5.0,"['Nuclear and High Energy Physics', 'Physics a...",unknown
7,10838880900,Morocho N.,Morocho,Nancy,N.,NaN,"['100556385', '100774304', '60072050']",7.0,"['Endocrinology', 'Obstetrics and Gynecology',...",unknown
8,10840143000,Bayas A.,Bayas,Antonio,A.,NaN,['60072054'],1.0,"['Electrical and Electronic Engineering', 'Com...",male
9,11141724800,Smalligan R.,Smalligan,Roger,R.,NaN,['60072047'],13.0,"['Leadership and Management', 'Epidemiology', ...",male


In [77]:
print(f"Autores con género asignado: {df_final.shape[0]}")
print(f"Autores sin información de género (NaN): {df_auths_without_gender["gender"].isna().sum()}")
print(
    f"Porcentaje completado: {(df_final.shape[0]) / df_authors.shape[0] * 100:.2f}%"
)

Autores con género asignado: 48252
Autores sin información de género (NaN): 17295
Porcentaje completado: 73.61%


In [67]:
df_final.to_csv("data/labels/autores_genero_imputado.csv", index=False)

In [80]:
df_genders_incomputed.to_csv("data/labels/autores_genero_no_imputado.csv", index=False)